# OVOS Intent Models: Score on the v6.1 Gold Split, Train One Locale

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/OpenVoiceOS/ngi0-notebooks/blob/dev/notebooks/ovos_intent_eval_and_train.ipynb) [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenVoiceOS/ngi0-notebooks/blob/dev/notebooks/ovos_intent_eval_and_train.ipynb)

[![NGI0 Commons Fund](../ngi.png)](https://nlnet.nl/project/OpenVoiceOS)

This project was funded through the NGI0 Commons Fund, a fund established by NLnet with financial support from the European Commission's Next Generation Internet programme, under the aegis of DG Communications Networks, Content and Technology under grant agreement No 101135429. Additional funding is made available by the Swiss State Secretariat for Education, Research and Innovation (SERI).

> This notebook was written with AI assistance (Claude, via Claude Code). Verify outputs before relying on them.

## Start here

**What it does.** An intent classifier decides what a person wants from a sentence such as "turn off the kitchen light". OpenVoiceOS publishes two such classifiers. This notebook downloads the hand-checked test sentences (the "gold split") of the OVOS intent corpus and measures how often each published classifier names the right intent, language by language. It prints what it measured and nothing else.

**What it is built on.** The repository's own build and training scripts are not yet released. This notebook therefore uses what is published: the corpus `OpenVoiceOS/ovos-intents` at tag `v6.1`, the models `OpenVoiceOS/ovos-m2v-intents-multilingual` and `OpenVoiceOS/ovos-m2v-intents-en`, and the `ovos-m2v-pipeline` package from PyPI, which is the code that loads those models inside OpenVoiceOS.

**How long.** Part 1 (scoring) runs on a CPU, with no GPU. The notebook prints its own wall time when it finishes. Part 2 (training one small language yourself) is off by default. It prints its own wall time and peak memory when you switch it on.

**What you must set.** Nothing for Part 1. Internet access must be on (Kaggle: Settings panel, Internet on). To also train a model, set `RUN_TRAINING = True` in the second cell. To try another language, change `TRAIN_LANG` there.

**What you will see at the end.** For every language in the gold split: the number of test sentences, the share the model got right, and the macro-F1 (the average of the per-intent F1 scores, so a rare intent counts as much as a common one). A note lists the intents the models were never trained on. Part 2, when on, adds the same numbers for the model you trained.

Press **Run All** (Kaggle: Run, then Run All; Colab: Runtime, then Run all).

Scores the published OVOS intent classifiers on the v6.1 gold split and, optionally, trains one small locale from the v6.1 training templates.

| Field | Value |
|---|---|
| Inputs | `OpenVoiceOS/ovos-intents` at tag `v6.1` (gold files `{lang}/test.jsonl`, training files `{lang}/train_templates.jsonl`); the models `OpenVoiceOS/ovos-m2v-intents-multilingual` and `OpenVoiceOS/ovos-m2v-intents-en`; for Part 2 the base model `minishlab/potion-base-32M` |
| Outputs | per-language accuracy and macro-F1 tables printed in the notebook; with `RUN_TRAINING = True`, a model folder and an ONNX file under `OUTPUT_DIR` |
| Runtime | measured by the notebook itself and printed at the end of each part |
| Hardware | CPU only |
| Cost | CPU only; downloads the two models and the gold files, and for Part 2 one training file |

CI: linted, compiled under Python 3.11 and 3.12, and executed on the smoke path. The smoke path scores a few sentences of three languages and does not train.

**Read more:**

- [Intents](https://openvoiceos.github.io/ovos-technical-manual/399-intents/)
- [ovos-m2v-pipeline](https://github.com/OpenVoiceOS/ovos-m2v-pipeline)
- [The corpus](https://huggingface.co/datasets/OpenVoiceOS/ovos-intents)
- [Model2Vec](https://github.com/MinishLab/model2vec)

In [ ]:
SMOKE = False            # True: quick check, three languages and N_SAMPLES test sentences each, no training.
N_SAMPLES = 4            # smoke only: test sentences scored per language
EPOCHS = 5               # Part 2 only: the most passes over the training rows (training stops early when it stops improving)
RUN_TRAINING = False     # True: also run Part 2 and train one small language
TRAIN_LANG = "gl-ES"     # Part 2: the language to train (needs {lang}/train_templates.jsonl in the corpus)
ROW_CAP = 20000          # Part 2: the most training rows used
MODEL_ID = "minishlab/potion-base-32M"  # Part 2: the small starting model the classifier is built on

## Cell 2: Settings

**Nothing needs editing: click Run All.** The first cell holds the switches. `RUN_TRAINING = True` turns on Part 2. `TRAIN_LANG` and `ROW_CAP` choose the language and how many training rows it may use. Every variable can also be provided as an environment variable with the same name.

In [ ]:
import os, time
T0 = time.time()

def _env(name, default):
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.lower() in ("1", "true", "yes") if isinstance(default, bool) else type(default)(raw)

SMOKE = _env("SMOKE", SMOKE)
N_SAMPLES = _env("N_SAMPLES", N_SAMPLES)
EPOCHS = _env("EPOCHS", EPOCHS)
RUN_TRAINING = _env("RUN_TRAINING", RUN_TRAINING)
TRAIN_LANG = _env("TRAIN_LANG", TRAIN_LANG)
ROW_CAP = _env("ROW_CAP", ROW_CAP)
MODEL_ID = _env("MODEL_ID", MODEL_ID)
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "./intent_output")

CORPUS = "OpenVoiceOS/ovos-intents"
CORPUS_TAG = "v6.1"
PUBLISHED_MODELS = ["OpenVoiceOS/ovos-m2v-intents-multilingual", "OpenVoiceOS/ovos-m2v-intents-en"]
SMOKE_LANGS = ["en-US", "gl-ES", "pt-PT"]
print(f"SMOKE={SMOKE} RUN_TRAINING={RUN_TRAINING} TRAIN_LANG={TRAIN_LANG} ROW_CAP={ROW_CAP}")

## Cell 3: Install dependencies

Installs the scoring package `ovos-m2v-pipeline` and the libraries around it. The package is published on PyPI as a pre-release, so the cell asks for pre-releases. If the install fails, check your network connection and run the cell again.

In [ ]:
%pip install --quiet --pre "ovos-m2v-pipeline>=0.33.0a1" "model2vec[train]>=0.9.0" "tokenizers>=0.20,<1.0.0" "datasets>=5.0.1" "huggingface_hub>=1.0" "scikit-learn>=1.9.0" "numpy>=2.0" "torch>=2.14.0" "onnx>=1.22.0" "onnxruntime>=1.30.0" "onnxscript>=0.7.1" "psutil>=7.0" "pandas>=2.2"

## Cell 4: Load the gold split

The corpus keeps one folder per language. A language has a file `{lang}/test.jsonl` only when someone wrote gold sentences for it, so the cell first lists which languages have one. It then reads each file at the fixed tag `v6.1`, so the numbers always refer to the same build of the corpus.

In [ ]:
import numpy as np, pandas as pd
from datasets import load_dataset
from huggingface_hub import list_repo_files

files = list_repo_files(CORPUS, repo_type="dataset", revision=CORPUS_TAG)
gold_langs = sorted(f.split("/")[0] for f in files if f.endswith("/test.jsonl"))
train_langs = sorted(f.split("/")[0] for f in files if f.endswith("/train_templates.jsonl"))
print(f"{CORPUS}@{CORPUS_TAG}: {len(gold_langs)} languages with a gold file, {len(train_langs)} with training templates")

if SMOKE:
    gold_langs = [l for l in SMOKE_LANGS if l in gold_langs]

gold = {}
for lang in gold_langs:
    ds = load_dataset(CORPUS, data_files=f"{lang}/test.jsonl", split="train", revision=CORPUS_TAG)
    if SMOKE:
        ds = ds.select(range(min(N_SAMPLES, len(ds))))
    gold[lang] = ds
print(f"gold rows loaded: {sum(len(d) for d in gold.values())} across {len(gold)} languages")
print("columns:", gold[gold_langs[0]].column_names)

## Cell 5: Load the published models

`ovos_m2v_pipeline.load_shared_model` is the call OpenVoiceOS itself makes to load a classifier, so the notebook scores the model the way a device would load it. Each model repository also carries a file `labels.json` that lists the intents the model was trained to name. The package also keeps a table, `RENAMED_LABELS`, of intents that a skill has renamed since a model was trained. At run time the pipeline maps a model's old name to the new one, so the notebook does the same and also prints the score without that mapping.

In [ ]:
import json
from huggingface_hub import hf_hub_download
from ovos_m2v_pipeline import load_shared_model
from ovos_m2v_pipeline.renames import RENAMED_LABELS

models = {}
for repo in PUBLISHED_MODELS:
    t = time.time()
    pipe = load_shared_model(repo, "classifier")
    valid = json.load(open(hf_hub_download(repo, "labels.json")))["valid_labels"]
    models[repo] = {"pipe": pipe, "classes": set(str(c) for c in pipe.classes_), "manifest": set(valid)}
    print(f"{repo}: {len(pipe.classes_)} intents, loaded in {time.time()-t:.1f} s")
print("renamed-intent table entries:", len(RENAMED_LABELS))

## Cell 6: Score every language

For each language the cell asks the model for one intent per gold sentence and compares it with the gold intent. **Accuracy** is the share of sentences where the two are equal. **Macro-F1** is computed over the intents that occur in that language's gold sentences, each counted equally. A gold intent the model was never trained on can never be predicted, so it counts as wrong; the next cell lists those.

**Renamed intents.** When a skill renames an intent after a model was trained, the model keeps naming the old spelling. The pipeline's rule at run time is: an entry of `RENAMED_LABELS` applies when the skill registers the new name and does not register the old one. Here, the intents the skills register are the intents of the v6.1 gold files. The same function is applied to the gold labels and to the predictions, so a spelling that a skill still registers (for example `Laugh`) is left alone on both sides.

In [ ]:
from sklearn.metrics import f1_score

def make_renamer(registered):
    """The pipeline's rule (Model2VecIntentPipeline._renamed_labels_to_route): rename when
    the new name is registered and the old one is not."""
    table = {old: new for old, new in RENAMED_LABELS.items() if new in registered and old not in registered}
    return lambda labels: [table.get(str(x), str(x)) for x in labels]

registered = set(l for ds in gold.values() for l in ds["label"])
mapped = make_renamer(registered)

results = {}   # repo -> DataFrame, one row per language
preds = {}
for repo, m in models.items():
    known = set(mapped(m["classes"]))
    rows = []
    for lang, ds in gold.items():
        y = mapped(ds["label"])
        raw = [str(p) for p in m["pipe"].predict(list(ds["utterance"]))]
        p = mapped(raw)
        preds[(repo, lang)] = (y, p)
        n_known = sum(g in known for g in y)
        in_known = [(g, q) for g, q in zip(y, p) if g in known]
        rows.append({
            "lang": lang,
            "rows": len(y),
            "rows_known": n_known,
            "accuracy": float(np.mean([a == b for a, b in zip(y, p)])),
            "macro_f1": float(f1_score(y, p, labels=sorted(set(y)), average="macro", zero_division=0)),
            "accuracy_unmapped": float(np.mean([a == b for a, b in zip(list(ds["label"]), raw)])),
            "accuracy_known": float(np.mean([a == b for a, b in in_known])) if in_known else float("nan"),
        })
    results[repo] = pd.DataFrame(rows)
print("scored", sum(len(v) for v in results.values()), "language/model pairs")

## Cell 6b: Self-checks

These checks fail the notebook, loudly, if scoring is wrong: every loaded row must be scored, every score must be a share between 0 and 1, at least one sentence must be scored correct, and the renaming rule must behave as described above.

In [ ]:
for repo, df in results.items():
    for _, r in df.iterrows():
        assert r["rows"] == len(gold[r["lang"]]), (repo, r["lang"], "rows scored != rows loaded")
        assert len(preds[(repo, r["lang"])][0]) == len(preds[(repo, r["lang"])][1]) == r["rows"]
        for col in ("accuracy", "macro_f1", "accuracy_unmapped"):
            assert 0.0 <= r[col] <= 1.0, (repo, r["lang"], col, r[col])
multi = PUBLISHED_MODELS[0]
n_right = sum(a == b for y_, p_ in (preds[(multi, l)] for l in gold) for a, b in zip(y_, p_))
assert n_right > 0, "the multilingual model scored no sentence correct: label normalisation or loading is broken"

old, new = "ovos-skill-laugh.openvoiceos:Laugh", "ovos-skill-laugh.openvoiceos:laugh"
assert RENAMED_LABELS[old] == new
assert make_renamer({new})([old]) == [new]             # only the new name is registered: rename
assert make_renamer({old, new})([old]) == [old]        # the skill still registers the old name: keep it
assert make_renamer({old})([old]) == [old]
print(f"self-checks passed; {n_right} of {sum(len(preds[(multi, l)][0]) for l in gold)} sentences correct for the multilingual model")

## Cell 7: Results per language

One block per model. `rows` is the number of gold sentences. `accuracy` and `macro_f1` use the renamed-intent mapping. The second table shows the accuracy without the mapping, how many gold sentences carry an intent the model cannot name (`rows_unknown`), and the accuracy on only the sentences it can name. Languages with few rows give a coarse number: one wrong sentence moves a 15-row language by 6.7 points.

In [ ]:
pd.set_option("display.width", 120)
for repo, df in results.items():
    print("=" * 60); print(repo); print("=" * 60)
    t1 = df[["lang", "rows", "accuracy", "macro_f1"]].copy()
    print(t1.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    tot = df["rows"].sum()
    micro = float(np.sum(df["accuracy"] * df["rows"]) / tot)
    print(f"all {tot} rows pooled: accuracy {micro:.3f}   mean of per-language macro-F1 {df['macro_f1'].mean():.3f}")
    t2 = df[["lang", "accuracy_unmapped"]].copy()
    t2["rows_unknown"] = df["rows"] - df["rows_known"]
    t2["accuracy_known"] = df["accuracy_known"]
    print("\nwithout the renamed-intent mapping, and on the sentences the model can name:")
    print(t2.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

## Cell 8: Intents the models never trained on

A classifier can only answer with an intent it was trained on. The cell counts, per model, the gold intents (after the renamed-intent mapping) that are not in the model's list, and the gold sentences that carry them. Those sentences count as wrong above. It also shows which published model can name the most gold intents; the English model was trained on English text only, so on other languages its numbers say little.

In [ ]:
gold_all = pd.concat([d.to_pandas() for d in gold.values()], ignore_index=True)
gold_all["label"] = mapped(gold_all["label"])
for repo, m in models.items():
    known = set(mapped(m["classes"]))
    unk = gold_all[~gold_all["label"].isin(known)]
    n_lab = gold_all["label"].nunique()
    print(f"{repo}")
    print(f"  gold intents in this run: {n_lab}; not in the model's list: {unk['label'].nunique()}; "
          f"gold sentences carrying one: {len(unk)} of {len(gold_all)}")
    for lab, cnt in unk["label"].value_counts().head(8).items():
        print(f"    {cnt:5d}  {lab}")
    extra = known - set(gold_all["label"])
    print(f"  intents the model can name that no gold sentence in this run uses: {len(extra)}")

## Cell 9: Part 1 finished

Part 1 is complete when the next cell prints its wall time. Everything above is a measurement of this run on this machine and this corpus tag; nothing in it is copied from a model card.

## Part 2 (optional): train one small language

Part 2 builds a new classifier for one language from the corpus training templates. It is off unless `RUN_TRAINING = True`. In order, it:

1. reads `{lang}/train_templates.jsonl` at `v6.1` for the language in `TRAIN_LANG`;
2. keeps at most `ROW_CAP` rows, spread evenly over the intents so rare intents stay in;
3. trains a Model2Vec `StaticModelForClassification` on top of `minishlab/potion-base-32M` (the `MODEL_ID` setting);
4. scores it on the same gold sentences for that language, with the same two measures;
5. saves it, and exports it to ONNX if that works;
6. prints how long the whole of Part 2 took and the most memory it used.

The conversion from the corpus rows to training lists (one sentence per row, one intent name per row) is done in the cells below; no script from the repository is used.

In [ ]:
print(f"Part 1 wall time: {time.time()-T0:.1f} s")

In [ ]:
if RUN_TRAINING:
    import threading, psutil, resource
    if TRAIN_LANG not in train_langs:
        raise SystemExit(f"{TRAIN_LANG} has no train_templates.jsonl in {CORPUS}@{CORPUS_TAG}")

    class PeakRss:
        """Samples this process's resident memory while Part 2 runs."""
        def __enter__(self):
            self.proc = psutil.Process()
            self.peak = self.proc.memory_info().rss
            self.stop = threading.Event()
            self.t = threading.Thread(target=self._run, daemon=True)
            self.t.start()
            return self
        def _run(self):
            while not self.stop.is_set():
                self.peak = max(self.peak, self.proc.memory_info().rss)
                time.sleep(0.1)
        def __exit__(self, *a):
            self.stop.set(); self.t.join()
            self.peak = max(self.peak, self.proc.memory_info().rss)

    P2_START = time.time()
    meter = PeakRss().__enter__()
    rss_before = meter.proc.memory_info().rss
else:
    print("RUN_TRAINING is False: Part 2 is skipped.")

In [ ]:
if RUN_TRAINING:
    t = time.time()
    tr = load_dataset(CORPUS, data_files=f"{TRAIN_LANG}/train_templates.jsonl", split="train", revision=CORPUS_TAG).to_pandas()
    print(f"{TRAIN_LANG}: {len(tr)} training rows, {tr['label'].nunique()} intents, read in {time.time()-t:.1f} s")
    cap = 200 if SMOKE else ROW_CAP
    per_label = max(1, cap // tr["label"].nunique())
    shuffled = tr.sample(frac=1.0, random_state=42)
    sel = shuffled[shuffled.groupby("label").cumcount() < per_label]
    if len(sel) < cap:   # intents shorter than the even share leave room: refill from the rest at random
        rest = shuffled.drop(sel.index)
        sel = pd.concat([sel, rest.iloc[: cap - len(sel)]])
    sel = sel.sample(frac=1.0, random_state=42)
    X_train, y_train = [str(x) for x in sel["utterance"]], [str(x) for x in sel["label"]]
    print(f"rows kept: {len(X_train)} (cap {cap}), intents kept: {len(set(y_train))}")
    gold_lang = gold.get(TRAIN_LANG)
    if gold_lang is None:
        print(f"note: {TRAIN_LANG} has no gold file in this run, so the trained model cannot be scored")

In [ ]:
if RUN_TRAINING:
    from model2vec.train import StaticModelForClassification
    t = time.time()
    clf = StaticModelForClassification.from_pretrained(model_name=MODEL_ID)
    clf.fit(X_train, y_train, max_epochs=EPOCHS if not SMOKE else 1, device="cpu")
    train_seconds = time.time() - t
    print(f"trained on {len(X_train)} rows in {train_seconds:.1f} s; {len(clf.classes_)} intents")

In [ ]:
if RUN_TRAINING and gold_lang is not None:
    y = list(gold_lang["label"])
    p = [str(q) for q in clf.predict(list(gold_lang["utterance"]))]
    trained = set(clf.classes_)
    acc = float(np.mean([a == b for a, b in zip(y, p)]))
    f1 = float(f1_score(y, p, labels=sorted(set(y)), average="macro", zero_division=0))
    n_known = sum(g in trained for g in y)
    print(f"model trained here, {TRAIN_LANG} gold: rows {len(y)}, accuracy {acc:.3f}, macro-F1 {f1:.3f}")
    print(f"  gold rows whose intent this model was trained on: {n_known} of {len(y)}")
    for repo, df in results.items():
        r = df[df["lang"] == TRAIN_LANG]
        if len(r):
            r = r.iloc[0]
            print(f"  published {repo.split('/')[-1]}: accuracy {r['accuracy']:.3f}, macro-F1 {r['macro_f1']:.3f}")

In [ ]:
if RUN_TRAINING:
    import pathlib
    out = pathlib.Path(OUTPUT_DIR) / f"intent-{TRAIN_LANG}"
    out.mkdir(parents=True, exist_ok=True)
    clf.to_pipeline().save_pretrained(str(out))
    print("saved model2vec pipeline folder:", out, sorted(p.name for p in out.iterdir()))

    # model2vec has no ONNX exporter of its own; the classifier is a torch module whose
    # forward pass maps token ids to class scores, so that part is exported with torch.
    onnx_path = out / "classifier.onnx"
    import torch
    try:
        import torch.onnx
        import onnxruntime as ort
    except ImportError as e:
        raise SystemExit(f"ONNX export is not possible here, the ONNX packages are missing: {e}")
    clf.eval()
    ids = clf.tokenize(["turn on the light", "what is the weather"])
    torch.onnx.export(clf.cpu(), (ids,), str(onnx_path), input_names=["input_ids"], output_names=["logits"],
                      dynamic_axes={"input_ids": {0: "batch", 1: "tokens"}, "logits": {0: "batch"}},
                      dynamo=False)
    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    probe = clf.tokenize(list(gold_lang["utterance"])[:8] if gold_lang is not None else X_train[:8])
    got = sess.run(None, {"input_ids": probe.numpy()})[0]
    with torch.no_grad():
        want = clf(probe).numpy()
    diff = float(np.abs(got - want).max())
    assert onnx_path.stat().st_size > 0 and diff < 1e-4, f"ONNX output differs from the torch model by {diff:.2e}"
    assert (got.argmax(1) == want.argmax(1)).all(), "ONNX and torch models pick different intents"
    print(f"ONNX written: {onnx_path.name}, {onnx_path.stat().st_size/1e6:.1f} MB; "
          f"max difference to the torch model on 8 sentences: {diff:.2e} (limit 1e-4)")

In [ ]:
if RUN_TRAINING:
    meter.__exit__(None, None, None)
    wall = time.time() - P2_START
    print(f"Part 2 wall time: {wall:.1f} s")
    print(f"Part 2 peak memory (resident, sampled every 0.1 s): {meter.peak/2**20:.0f} MiB; "
          f"in use when Part 2 began: {rss_before/2**20:.0f} MiB")
    print(f"whole notebook process peak memory so far (kernel-reported): {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024:.0f} MiB")

## Reading the results

- The gold sentences come from the test files the skills themselves ship, not from the corpus split the models were trained on, so the numbers here can differ a lot from the figures on a model card.
- A model scores zero on a gold intent it was never trained on. The never-trained note above says how many sentences that affects.
- Languages with fewer than about 30 gold sentences give coarse numbers.
- The English model is only meaningful on English; its other rows show what happens when it is asked about a language it never saw.
- Part 2 trains a small model on a capped sample and is a demonstration of the steps, not a replacement for a published model.